In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import random
import glob

In [2]:
# # find all pickle files in the current directory using glob

# path = "/Users/ens/Library/Mobile Documents/com~apple~CloudDocs/Rupali/new"
# all_files = glob.glob(path + "/*.pkl")

# # create a list of dataframes
# li = []

# for filename in all_files:
#     df = pd.read_pickle(filename)
#     li.append(df)

# # concatenate the list of dataframes into one dataframe
# data = pd.concat(li, axis=0, ignore_index=True)

In [3]:
# load datafram from pickle
data = pd.read_pickle('/Users/rupali/Documents/MARL/marl/data/diplomacy-v1-27k-msgs/output-2022-10-28_01-27-35_worker-0_data.pkl')
batch_size = 100

In [4]:
def data_prep(data, batch_size, ratios=[0.0, 0.8, 0.9]):

    # do train test validation split on the data
    # the data is split into ratios[0:1] train, ratios[1:2] test, ratios[2:]validation
    # the data is shuffled before splitting

    df = data.groupby(['eps_id','t']).agg({'obs':lambda x: list(np.concatenate(x.values)),
                                            'actions':lambda col: col.tolist(),
                                            'agent_index':lambda col: col.tolist()}).reset_index()

    # shuffling by episode
    groups = [df for _, df in df.groupby('eps_id')]
    random.shuffle(groups)
    df = pd.concat(groups).reset_index(drop=True)

    # splitting dataset
    _, train, test, val = np.split(df, [int(ratios[0]*len(df)),int(ratios[1]*len(df)), int(ratios[2]*len(df))])

    print("train shape: ", train.shape)
    print("test shape: ", test.shape)
    print("val shape: ", val.shape)

    # 

    obs_train, obs_test, obs_val = np.array(train['obs'].to_list()), np.array(test['obs'].to_list()), np.array(val['obs'].to_list())
    act_train, act_test, act_val = np.array(train['actions'].to_list()), np.array(test['actions'].to_list()), np.array(val['actions'].to_list())
    ids_train, ids_test, ids_val = np.array(train['agent_index'].to_list()), np.array(test['agent_index'].to_list()), np.array(val['agent_index'].to_list())


    # create test, train and validation dataloaders
    # the dataloaders will be used to train the neural network
    # the dataloaders will return a batch of observations and actions
    # the batch size is set to 100
    # the shuffle parameter is set to True so that the data is shuffled before each epoch

    train_dataset = torch.utils.data.TensorDataset(torch.from_numpy(obs_train), torch.from_numpy(act_train))
    train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    test_dataset = torch.utils.data.TensorDataset(torch.from_numpy(obs_test), torch.from_numpy(act_test))
    test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

    val_dataset = torch.utils.data.TensorDataset(torch.from_numpy(obs_val), torch.from_numpy(act_val))
    val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

    return train_dataloader, test_dataloader, val_dataloader

In [5]:
# def data_prep(data, batch_size, ratios=[0.0, 0.8, 0.9]):

#     # do train test validation split on the data
#     # the data is split into ratios[0:1] train, ratios[1:2] test, ratios[2:]validation
#     # the data is shuffled before splitting

#     _, train, test, val = np.split(data.sample(frac=1), [int(ratios[0]*len(data)),int(ratios[1]*len(data)), int(ratios[2]*len(data))])

#     print("train shape: ", train.shape)
#     print("test shape: ", test.shape)
#     print("val shape: ", val.shape)

#     # 

#     obs_train, obs_test, obs_val = np.array(train['obs'].to_list()), np.array(test['obs'].to_list()), np.array(val['obs'].to_list())
#     act_train, act_test, act_val = np.array(train['actions'].to_list()), np.array(test['actions'].to_list()), np.array(val['actions'].to_list())
#     ids_train, ids_test, ids_val = np.array(train['agent_index'].to_list()), np.array(test['agent_index'].to_list()), np.array(val['agent_index'].to_list())


#     # create test, train and validation dataloaders
#     # the dataloaders will be used to train the neural network
#     # the dataloaders will return a batch of observations and actions
#     # the batch size is set to 100
#     # the shuffle parameter is set to True so that the data is shuffled before each epoch

#     train_dataset = torch.utils.data.TensorDataset(torch.from_numpy(obs_train), torch.from_numpy(act_train))
#     train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

#     test_dataset = torch.utils.data.TensorDataset(torch.from_numpy(obs_test), torch.from_numpy(act_test))
#     test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

#     val_dataset = torch.utils.data.TensorDataset(torch.from_numpy(obs_val), torch.from_numpy(act_val))
#     val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

#     return train_dataloader, test_dataloader, val_dataloader

In [6]:
# get dataloaders for train, test and validation data
train_dataloader, test_dataloader, val_dataloader = data_prep(data, batch_size)

train shape:  (28040, 5)
test shape:  (3505, 5)
val shape:  (3505, 5)
